In [4]:
#!/usr/bin/env python3
"""
Workforce Analysis API Test Suite - April 2026
Tests all 5 workforce endpoints and validates responses
"""

import requests
import json
import sys
from datetime import datetime
from typing import Dict, Any, List
import pandas as pd
from tabulate import tabulate

# Configuration
BASE_URL = "http://localhost:8000"
FISCAL_PERIOD = "2026-04"
ENTITY_CODE = "AUS01"

class Colors:
    """ANSI color codes for terminal output"""
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    END = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

def print_header(text: str):
    """Print formatted header"""
    print(f"\n{Colors.CYAN}{'='*60}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.HEADER}{text}{Colors.END}")
    print(f"{Colors.CYAN}{'='*60}{Colors.END}\n")

def print_success(text: str):
    """Print success message"""
    print(f"{Colors.GREEN}✅ {text}{Colors.END}")

def print_error(text: str):
    """Print error message"""
    print(f"{Colors.RED}❌ {text}{Colors.END}")

def print_warning(text: str):
    """Print warning message"""
    print(f"{Colors.YELLOW}⚠️ {text}{Colors.END}")

def print_info(text: str):
    """Print info message"""
    print(f"{Colors.BLUE}📊 {text}{Colors.END}")

def test_endpoint(endpoint: str, params: Dict = None, expected_keys: List[str] = None) -> Dict:
    """
    Generic function to test an endpoint
    
    Args:
        endpoint: API endpoint path
        params: Query parameters
        expected_keys: List of expected keys in response data
    
    Returns:
        Response data as dictionary
    """
    url = f"{BASE_URL}{endpoint}"
    
    try:
        print_info(f"Calling: {url}")
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if data.get('success'):
            print_success(f"Endpoint returned successfully")
            
            # Validate expected keys if provided
            if expected_keys and 'data' in data:
                missing_keys = [key for key in expected_keys if key not in data['data']]
                if missing_keys:
                    print_warning(f"Missing expected keys: {missing_keys}")
                else:
                    print_success(f"All expected keys present: {expected_keys}")
            
            return data
        else:
            print_error(f"API returned success=False: {data.get('message', 'Unknown error')}")
            return None
            
    except requests.exceptions.ConnectionError:
        print_error(f"Cannot connect to {BASE_URL}. Is the application running?")
        return None
    except requests.exceptions.Timeout:
        print_error(f"Request timeout for {endpoint}")
        return None
    except requests.exceptions.HTTPError as e:
        print_error(f"HTTP Error {e.response.status_code}: {e.response.text[:200]}")
        return None
    except Exception as e:
        print_error(f"Unexpected error: {str(e)}")
        return None

def test_cost_revenue_correlation():
    """Test 1: Cost-Revenue Correlation by Business Unit"""
    print_header("TEST 1: Cost-Revenue Correlation by Business Unit")
    
    data = test_endpoint(
        "/tools/workforce/cost_revenue_correlation",
        params={"fiscal_period": FISCAL_PERIOD},
        expected_keys=["by_business_unit", "total_workforce_cost", "total_revenue_supported"]
    )
    
    if data and 'data' in data:
        result = data['data']
        
        # Display summary
        print_info(f"Total Workforce Cost: ${result.get('total_workforce_cost', 0):,.2f}")
        print_info(f"Total Revenue Supported: ${result.get('total_revenue_supported', 0):,.2f}")
        
        # Display by business unit in table format
        if 'by_business_unit' in result:
            print_info("\nBusiness Unit Performance:")
            
            table_data = []
            for bu, metrics in result['by_business_unit'].items():
                table_data.append([
                    bu,
                    f"${metrics.get('total_workforce_cost', 0):,.0f}",
                    f"${metrics.get('total_revenue_supported', 0):,.0f}",
                    f"{metrics.get('cost_per_revenue', 0)*100:.1f}%",
                    f"{metrics.get('revenue_per_employee', 0):,.0f}"
                ])
            
            headers = ["Business Unit", "Cost", "Revenue", "Cost/Revenue %", "Revenue/Employee"]
            print(tabulate(table_data, headers=headers, tablefmt="grid"))
        
        return True
    return False

def test_salary_analysis():
    """Test 2: Salary Analysis by Department"""
    print_header("TEST 2: Salary Analysis by Department")
    
    data = test_endpoint(
        "/tools/workforce/salary_analysis",
        params={"fiscal_period": FISCAL_PERIOD},
        expected_keys=["by_department", "total_risk_items", "risk_items"]
    )
    
    if data and 'data' in data:
        result = data['data']
        
        print_info(f"Total Risk Items: {result.get('total_risk_items', 0)}")
        
        # Display department summary
        if 'by_department' in result:
            print_info("\nDepartment Analysis:")
            
            table_data = []
            for dept, metrics in result['by_department'].items():
                table_data.append([
                    dept,
                    f"${metrics.get('total_salary', 0):,.0f}",
                    f"{metrics.get('overtime_percent', 0):.1f}%",
                    metrics.get('headcount', 0),
                    metrics.get('risk_items', 0),
                    f"{metrics.get('risk_percent', 0):.1f}%"
                ])
            
            headers = ["Department", "Total Salary", "Overtime %", "Headcount", "Risks", "Risk %"]
            print(tabulate(table_data, headers=headers, tablefmt="grid"))
        
        # Display risk items
        risk_items = result.get('risk_items', [])
        if risk_items:
            print_warning(f"\n⚠️ Risk Items Found ({len(risk_items)}):")
            for item in risk_items[:5]:  # Show first 5
                print(f"   • {item.get('employee_id')} - {item.get('role')} - {item.get('risk_flag')}")
                print(f"     Action: {item.get('recommended_action', 'Review required')}")
        
        return True
    return False

def test_cost_output_efficiency():
    """Test 3: Cost Output Efficiency Analysis"""
    print_header("TEST 3: Cost Output Efficiency Analysis")
    
    data = test_endpoint(
        "/tools/workforce/cost_output_efficiency",
        params={"fiscal_period": FISCAL_PERIOD},
        expected_keys=["inefficient_items", "by_business_unit", "total_inefficient_count"]
    )
    
    if data and 'data' in data:
        result = data['data']
        
        print_info(f"Total Inefficient Items: {result.get('total_inefficient_count', 0)}")
        
        # Display business unit efficiency
        if 'by_business_unit' in result:
            print_info("\nBusiness Unit Efficiency:")
            
            table_data = []
            for bu, metrics in result['by_business_unit'].items():
                table_data.append([
                    bu,
                    f"${metrics.get('total_cost', 0):,.0f}",
                    f"{metrics.get('average_utilisation', 0):.1f}%",
                    f"${metrics.get('cost_per_output', 0):,.2f}",
                    metrics.get('efficiency_rating', 'N/A')
                ])
            
            headers = ["Business Unit", "Total Cost", "Avg Utilisation", "Cost/Output", "Rating"]
            print(tabulate(table_data, headers=headers, tablefmt="grid"))
        
        # Display inefficient items
        inefficient_items = result.get('inefficient_items', [])
        if inefficient_items:
            print_warning(f"\n⚠️ Inefficient Items ({len(inefficient_items)}):")
            for item in inefficient_items[:5]:
                print(f"   • {item.get('employee_id')} - {item.get('role')}")
                print(f"     Utilisation: {item.get('utilisation_percent', 0)}% | "
                      f"Cost/Output: ${item.get('cost_per_output', 0):,.2f}")
        
        return True
    return False

def test_labour_cost_metrics():
    """Test 4: Labour Cost Metrics"""
    print_header("TEST 4: Labour Cost Metrics")
    
    data = test_endpoint(
        "/tools/workforce/labour_cost_metrics",
        params={"fiscal_period": FISCAL_PERIOD},
        expected_keys=["by_business_unit", "individual_metrics"]
    )
    
    if data and 'data' in data:
        result = data['data']
        
        # Display business unit metrics
        if 'by_business_unit' in result:
            print_info("\nBusiness Unit Labour Metrics:")
            
            table_data = []
            for bu, metrics in result['by_business_unit'].items():
                table_data.append([
                    bu,
                    f"${metrics.get('total_workforce_cost', 0):,.0f}",
                    f"${metrics.get('average_labour_cost_per_output', 0):,.2f}",
                    f"${metrics.get('min_labour_cost_per_output', 0):,.2f}",
                    f"${metrics.get('max_labour_cost_per_output', 0):,.2f}",
                    metrics.get('employee_count', 0)
                ])
            
            headers = ["Business Unit", "Total Cost", "Avg Cost/Output", "Min", "Max", "Employees"]
            print(tabulate(table_data, headers=headers, tablefmt="grid"))
        
        # Display top/bottom performers
        individual_metrics = result.get('individual_metrics', [])
        if individual_metrics:
            # Sort by cost per output
            sorted_metrics = sorted(individual_metrics, key=lambda x: x.get('labour_cost_per_output', 0))
            
            print_info("\n🏆 Top 5 Most Efficient Employees (Lowest Cost/Output):")
            for metric in sorted_metrics[:5]:
                print(f"   • {metric.get('employee_id')} - {metric.get('role')}: "
                      f"${metric.get('labour_cost_per_output', 0):,.2f} per output")
            
            print_info("\n⚠️ Bottom 5 Least Efficient Employees (Highest Cost/Output):")
            for metric in sorted_metrics[-5:]:
                print(f"   • {metric.get('employee_id')} - {metric.get('role')}: "
                      f"${metric.get('labour_cost_per_output', 0):,.2f} per output")
        
        return True
    return False

def test_resource_optimization():
    """Test 5: Resource Optimization Recommendations"""
    print_header("TEST 5: Resource Optimization Recommendations")
    
    data = test_endpoint(
        "/tools/workforce/resource_optimization",
        params={"fiscal_period": FISCAL_PERIOD},
        expected_keys=["recommendations", "total_potential_margin_improvement", "total_potential_savings"]
    )
    
    if data and 'data' in data:
        result = data['data']
        
        print_info(f"Total Recommendations: {len(result.get('recommendations', []))}")
        print_info(f"Potential Margin Improvement: ${result.get('total_potential_margin_improvement', 0):,.2f}")
        print_info(f"Potential Savings: ${result.get('total_potential_savings', 0):,.2f}")
        
        # Display recommendations
        recommendations = result.get('recommendations', [])
        if recommendations:
            print_info("\n📋 Recommendations:")
            
            for i, rec in enumerate(recommendations[:8], 1):
                print(f"\n   {i}. {rec.get('business_unit')} - {rec.get('role')}")
                print(f"      Issue: {rec.get('issue')}")
                print(f"      Current: {rec.get('current_utilisation', rec.get('current_cost_per_output', 'N/A'))}")
                print(f"      Action: {rec.get('action')}")
        
        return True
    return False

def test_consolidated_summary():
    """Bonus: Test Consolidated IBM BOB Summary"""
    print_header("BONUS: Consolidated IBM BOB Summary")
    
    data = test_endpoint(
        "/tools/ibm_bob/summary/2026-04",
        params={},
        expected_keys=["pl_metrics", "workforce_metrics", "esg_metrics", "readiness_score"]
    )
    
    if data and 'data' in data:
        result = data['data']
        
        print_info(f"Overall Readiness Score: {result.get('readiness_score', 0):.1f}%")
        print_info(f"Readiness Status: {result.get('readiness_status', 'UNKNOWN')}")
        
        # Display workforce metrics
        if 'workforce_metrics' in result:
            wf = result['workforce_metrics']
            print_info("\nWorkforce Summary:")
            print(f"   Total Employees: {wf.get('total_employees', 0)}")
            print(f"   Total Cost: ${wf.get('total_workforce_cost', 0):,.2f}")
            print(f"   Revenue Supported: ${wf.get('total_revenue_supported', 0):,.2f}")
            print(f"   Risk Items: {wf.get('risk_items', 0)}")
            print(f"   Average Utilisation: {wf.get('average_utilisation', 0):.1f}%")
        
        return True
    return False

def check_approval_items():
    """Check pending approval items in dashboard"""
    print_header("APPROVAL ITEMS CHECK")
    
    try:
        response = requests.get(f"{BASE_URL}/approvals/pending", timeout=10)
        response.raise_for_status()
        data = response.json()
        
        pending_count = data.get('count', 0)
        
        if pending_count > 0:
            print_warning(f"Found {pending_count} pending approval items")
            
            # Display first few pending items
            for item in data.get('data', [])[:5]:
                print(f"   • {item.get('type')}: {item.get('description')[:50]}...")
                print(f"     Amount: ${item.get('amount', 0):,.2f}")
                print(f"     Token: {item.get('token')[:16]}...")
        else:
            print_success("No pending approval items")
        
        return pending_count
        
    except Exception as e:
        print_error(f"Failed to check approvals: {str(e)}")
        return -1

def generate_report(test_results: Dict[str, bool]):
    """Generate final test report"""
    print_header("TEST SUMMARY REPORT")
    
    total_tests = len(test_results)
    passed_tests = sum(1 for v in test_results.values() if v)
    failed_tests = total_tests - passed_tests
    
    print(f"Test Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Environment: {BASE_URL}")
    print(f"Fiscal Period: {FISCAL_PERIOD}")
    print(f"\nResults:")
    print(f"  Total Tests: {total_tests}")
    print(f"  Passed: {passed_tests}")
    print(f"  Failed: {failed_tests}")
    print(f"  Success Rate: {(passed_tests/total_tests)*100:.1f}%")
    
    print(f"\nDetailed Results:")
    for test_name, result in test_results.items():
        status = "✅ PASSED" if result else "❌ FAILED"
        color = Colors.GREEN if result else Colors.RED
        print(f"  {color}{status}{Colors.END} - {test_name}")
    
    if failed_tests == 0:
        print_success(f"\n🎉 All tests passed! The system is ready for production use.")
    else:
        print_warning(f"\n⚠️ {failed_tests} test(s) failed. Please review the errors above.")
    
    print(f"\n{Colors.CYAN}{'='*60}{Colors.END}")

def main():
    """Main test execution"""
    print_header("WORKFORCE ANALYSIS API TEST SUITE - APRIL 2026")
    
    print(f"Testing against: {BASE_URL}")
    print(f"Fiscal Period: {FISCAL_PERIOD}")
    print(f"Entity: {ENTITY_CODE}")
    
    # Test connectivity first
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=5)
        if response.status_code == 200:
            print_success("Connected to Finance Month-End Close API")
        else:
            print_error(f"API health check failed with status {response.status_code}")
            sys.exit(1)
    except Exception as e:
        print_error(f"Cannot connect to API at {BASE_URL}")
        print_error(f"Make sure the application is running: python app.py")
        sys.exit(1)
    
    # Run all tests
    results = {}
    
    results['Cost-Revenue Correlation'] = test_cost_revenue_correlation()
    results['Salary Analysis'] = test_salary_analysis()
    results['Cost Output Efficiency'] = test_cost_output_efficiency()
    results['Labour Cost Metrics'] = test_labour_cost_metrics()
    results['Resource Optimization'] = test_resource_optimization()
    results['Consolidated Summary'] = test_consolidated_summary()
    
    # Check approval items (non-critical for API tests)
    pending = check_approval_items()
    results['Approval Items Check'] = pending >= 0
    
    # Generate final report
    generate_report(results)
    
    # Return exit code based on test results
    if all(results.values()):
        print_success("\n✅ All critical tests passed!")
        sys.exit(0)
    else:
        print_error("\n❌ Some tests failed. Please investigate the errors above.")
        sys.exit(1)

if __name__ == "__main__":
    main()


WORKFORCE ANALYSIS API TEST SUITE - APRIL 2026

Testing against: http://localhost:8000
Fiscal Period: 2026-04
Entity: AUS01
✅ Connected to Finance Month-End Close API

TEST 1: Cost-Revenue Correlation by Business Unit

📊 Calling: http://localhost:8000/tools/workforce/cost_revenue_correlation
✅ Endpoint returned successfully
✅ All expected keys present: ['by_business_unit', 'total_workforce_cost', 'total_revenue_supported']
📊 Total Workforce Cost: $14,327,190.00
📊 Total Revenue Supported: $174,550,000.00
📊 
Business Unit Performance:
+-----------------+------------+-------------+------------------+--------------------+
| Business Unit   | Cost       | Revenue     | Cost/Revenue %   |   Revenue/Employee |
+=================+============+=============+==================+====================+
| QLD             | $3,582,740 | $52,200,000 | 6.9%             |          2,747,368 |
+-----------------+------------+-------------+------------------+--------------------+
| NSW             | $3,57

SystemExit: 1